# BERTopic 主题挖掘（3 万条 · 云 GPU）

**环境**：Linux + NVIDIA（如 RTX 5090 32GB）。在**仓库根目录**打开本 Notebook。

**依赖**（终端一次性执行）：

```bash
cd <仓库根目录>
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -U pip
python -m pip install -r ml/requirements-bertopic.txt
```

**语料**：默认 `ml/topic_mining/data/topic_subsets/topic_input_30k_seed42.csv`，文本列 `analysis_input_en`。

**防断连**：可用 `tmux` / `screen` 后台跑；第一次会从 Hugging Face 下载句向量权重（需网络）。

## 可选：一键拉仓库（重新 Git 场景）

如果你在 AutoDL 上是全新环境，先运行下一格自动拉取仓库到 `/root/autodl-tmp/RSA`。

- 已存在同名目录会自动跳过，不会覆盖。
- 若你的仓库地址不同，改 `REPO_URL` 即可。

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/KevinL35/RSA.git"
REPO_PARENT = Path("/root/autodl-tmp")
REPO_DIR = REPO_PARENT / "RSA"

REPO_PARENT.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    print("已克隆:", REPO_DIR)
else:
    print("已存在，跳过克隆:", REPO_DIR)

print("仓库目录:", REPO_DIR)

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
repo_from_bootstrap = globals().get("REPO_DIR")
if repo_from_bootstrap and (Path(repo_from_bootstrap) / "ml" / "topic_mining").is_dir():
    ROOT = Path(repo_from_bootstrap)
elif (cwd / "ml" / "topic_mining").is_dir():
    ROOT = cwd
elif (Path("/root/autodl-tmp/RSA") / "ml" / "topic_mining").is_dir():
    ROOT = Path("/root/autodl-tmp/RSA")
else:
    raise RuntimeError(
        "未找到仓库根目录。请先运行上一格自动 git clone，或在 RSA 根目录打开 Notebook。"
    )

PYTHON = ROOT / ".venv" / "bin" / "python"
if not PYTHON.is_file():
    PYTHON = Path(sys.executable)

DATA_CSV = ROOT / "ml/topic_mining/data/topic_subsets/topic_input_30k_seed42.csv"
TEXT_COL = "analysis_input_en"
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

ARTIFACT_DIR = ROOT / "ml/topic_mining/artifacts/bertopic_30k_bge_seed42_v0"
REPORT_DIR = ROOT / "ml/topic_mining/reports"
REPORT_JSON = REPORT_DIR / "bertopic_30k_run_meta.json"
TOPIC_INFO_CSV = REPORT_DIR / "bertopic_30k_topic_info.csv"
DOC_TOPICS_CSV = REPORT_DIR / "bertopic_30k_doc_topics_sample.csv"

RANDOM_STATE = 42
EMBED_BATCH = 128

print("ROOT:", ROOT)
print("PYTHON:", PYTHON)
print("DATA_CSV exists:", DATA_CSV.is_file())

## 0) 可选：在此环境中补装依赖

若已按上文在终端 `pip install -r ml/requirements-bertopic.txt`，可跳过本格。

In [ ]:
req = ROOT / "ml/requirements-bertopic.txt"
if req.is_file():
    # 用当前 Notebook 内核对应的 Python 安装依赖，避免“装在 .venv 但内核不是 .venv”导致导入失败。
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req)], check=True)
    print("已在当前内核安装依赖:", sys.executable)
    if Path(sys.executable).resolve() != PYTHON.resolve():
        print("提示：当前内核与 .venv 不一致；已优先保证当前内核可用。")
else:
    print("未找到", req)

## 1) GPU 与 PyTorch

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 2) 读入语料并去空

In [ ]:
if not DATA_CSV.is_file():
    raise FileNotFoundError(f"缺少数据文件: {DATA_CSV}")

df = pd.read_csv(DATA_CSV, encoding="utf-8")
if TEXT_COL not in df.columns:
    raise ValueError(f"缺少列 {TEXT_COL!r}，当前列: {list(df.columns)}")

s = df[TEXT_COL].astype(str).str.strip()
df = df[s != ""].copy().reset_index(drop=True)

docs = df[TEXT_COL].tolist()
print("rows:", len(df))
if len(df) != 30000:
    print("警告：当前不是 30000 条，确认是否仍使用 30k 子集文件。")
print(df.head(2))

## 3) 句向量 + BERTopic（强基座 bge-base-en）

- **UMAP / HDBSCAN** 默认在 CPU 上跑；3 万条一般几分钟级可接受。
- 句向量在 **GPU** 上计算（`SentenceTransformer`）。
- `normalize_embeddings=True` 与 BGE 聚类常用设定一致。

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import umap

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device=device, trust_remote_code=True)

_encode_orig = embedding_model.encode


def encode_normalized(sentences, **kwargs):
    kwargs.setdefault("batch_size", EMBED_BATCH)
    kwargs.setdefault("normalize_embeddings", True)
    kwargs.setdefault("show_progress_bar", True)
    return _encode_orig(sentences, **kwargs)


embedding_model.encode = encode_normalized

vectorizer_model = CountVectorizer(
    min_df=2,
    ngram_range=(1, 2),
    stop_words="english",
)

umap_model = umap.UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    min_topic_size=30,
    nr_topics=None,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs)
print("n_topics (incl. -1):", len(set(topic_model.get_topics().keys())))
topic_model.get_topic_info().head(5)

## 4) 导出主题表、抽样文档主题、保存模型

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.parent.mkdir(parents=True, exist_ok=True)

info = topic_model.get_topic_info()
info.to_csv(TOPIC_INFO_CSV, index=False)
print("topic_info ->", TOPIC_INFO_CSV)

out = df.copy()
out["topic"] = topics
sample_n = min(2000, len(out))
out.sample(sample_n, random_state=RANDOM_STATE).to_csv(DOC_TOPICS_CSV, index=False)
print("doc_topics_sample ->", DOC_TOPICS_CSV)

# 不内嵌句向量文件以节省空间；加载时需再传同一 EMBEDDING_MODEL
topic_model.save(
    str(ARTIFACT_DIR),
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=False,
)

meta = {
    "data_csv": str(DATA_CSV.relative_to(ROOT)),
    "n_rows": len(df),
    "text_column": TEXT_COL,
    "embedding_model": EMBEDDING_MODEL,
    "device": device,
    "random_state": RANDOM_STATE,
    "artifact_dir": str(ARTIFACT_DIR.relative_to(ROOT)),
    "n_topic_ids": int(len(set(topics))),
    "outlier_topic_rows": int((out["topic"] == -1).sum()),
}
REPORT_JSON.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("meta ->", REPORT_JSON)
print("model saved ->", ARTIFACT_DIR)

## 5) 重新加载示例（另一进程/Notebook）

```python
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

emb = SentenceTransformer("BAAI/bge-base-en-v1.5", trust_remote_code=True)
model = BERTopic.load(
    "ml/topic_mining/artifacts/bertopic_30k_bge_seed42_v0",
    embedding_model=emb,
)
```

（注意：`embedding_model` 必须与训练时一致。）